In [ ]:
from pyspark.sql.functions import col, trim, lower, upper, regexp_replace, to_date, to_timestamp
import os
from pyspark.sql import SparkSession

os.environ["AWS_REGION"] = "us-east-1"
os.environ["AWS_ACCESS_KEY_ID"] = "admin"
os.environ["AWS_SECRET_ACCESS_KEY"] = "password"

iceberg_version = "1.4.3"
aws_version = "3.3.4"
packages = [
    f"org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:{iceberg_version}",
    f"org.apache.iceberg:iceberg-aws-bundle:{iceberg_version}",
    f"org.apache.hadoop:hadoop-aws:{aws_version}",
    "com.amazonaws:aws-java-sdk-bundle:1.12.262"
]

spark = SparkSession.builder \
    .appName("Ingestao-UFMT-Prata") \
    .config("spark.jars.packages", ",".join(packages)) \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.iceberg", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.iceberg.type", "hive") \
    .config("spark.sql.catalog.iceberg.uri", "thrift://hive-metastore:9083") \
    .config("spark.sql.catalog.iceberg.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.iceberg.warehouse", "s3a://warehouse/") \
    .config("spark.sql.catalog.iceberg.s3.endpoint", "http://minio:9000") \
    .config("spark.sql.catalog.iceberg.client.region", "us-east-1") \
    .config("spark.sql.catalog.iceberg.s3.path-style-access", "true") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
spark.sql("CREATE NAMESPACE IF NOT EXISTS iceberg.silver")

print("Lendo da camada Bronze (CEIS Bruto)...")
df_bronze = spark.table("iceberg.bronze.ceis_raw")

print("Aplicando limpeza + tipagem (Camada Prata)...")
df_silver = (
    df_bronze
    .select(
        col("CADASTRO").alias("cadastro"),
        col("CÓDIGO DA SANÇÃO").alias("codigo_sancao"),
        col("TIPO DE PESSOA").alias("tipo_pessoa"),
        col("CPF OU CNPJ DO SANCIONADO").alias("cpf_cnpj"),
        col("NOME DO SANCIONADO").alias("nome_sancionado"),
        col("CATEGORIA DA SANÇÃO").alias("categoria_sancao"),
        col("DATA INÍCIO SANÇÃO").alias("data_inicio_sancao"),
        col("DATA FINAL SANÇÃO").alias("data_final_sancao"),
        col("DATA PUBLICAÇÃO").alias("data_publicacao"),
        col("ÓRGÃO SANCIONADOR").alias("orgao_sancionador"),
        col("UF ÓRGÃO SANCIONADOR").alias("uf_orgao_sancionador"),
        col("ESFERA ÓRGÃO SANCIONADOR").alias("esfera_orgao_sancionador"),
        col("NÚMERO DO PROCESSO").alias("numero_processo"),
        col("ABRAGÊNCIA DA SANÇÃO").alias("abrangencia_sancao"),
        col("FUNDAMENTAÇÃO LEGAL").alias("fundamentacao_legal"),
        col("OBSERVAÇÕES").alias("observacoes"),
        col("data_ingestao").alias("data_ingestao")
    )
    # limpeza
    .withColumn("cadastro", upper(trim(col("cadastro"))))
    .withColumn("tipo_pessoa", upper(trim(col("tipo_pessoa"))))
    .withColumn("cpf_cnpj", regexp_replace(trim(col("cpf_cnpj")), "[^0-9]", ""))
    .withColumn("nome_sancionado", trim(col("nome_sancionado")))
    .withColumn("nome_sancionado_norm", lower(trim(col("nome_sancionado"))))
    .withColumn("categoria_sancao", trim(col("categoria_sancao")))
    .withColumn("orgao_sancionador", trim(col("orgao_sancionador")))
    .withColumn("uf_orgao_sancionador", upper(trim(col("uf_orgao_sancionador"))))
    .withColumn("esfera_orgao_sancionador", upper(trim(col("esfera_orgao_sancionador"))))
    .withColumn("numero_processo", trim(col("numero_processo")))
    .withColumn("abrangencia_sancao", trim(col("abrangencia_sancao")))
    .withColumn("fundamentacao_legal", trim(col("fundamentacao_legal")))
    .withColumn("observacoes", trim(col("observacoes")))
    # tipagem
    .withColumn("codigo_sancao", col("codigo_sancao").cast("int"))
    .withColumn("data_inicio_sancao", to_date(col("data_inicio_sancao"), "dd/MM/yyyy"))
    .withColumn("data_final_sancao", to_date(col("data_final_sancao"), "dd/MM/yyyy"))
    .withColumn("data_publicacao", to_date(col("data_publicacao"), "dd/MM/yyyy"))
    .withColumn("data_ingestao", to_timestamp(col("data_ingestao")))
    # filtros
    .filter(col("cpf_cnpj").isNotNull() & (col("cpf_cnpj") != ""))
    .filter(col("nome_sancionado").isNotNull() & (col("nome_sancionado") != ""))
)

print("Gravando na camada Prata...")
df_silver.writeTo("iceberg.silver.ceis_cleansed") \
    .tableProperty("format-version", "2") \
    .using("iceberg") \
    .createOrReplace()

print("Prata finalizada com limpeza e tipagem.")

In [ ]:
df_show = spark.sql("SELECT * FROM iceberg.silver.ceis_cleansed LIMIT 10")
df_show.show(truncate=False)